In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import timm
import time

In [ ]:
# cifar_dataset_path = "./cifar10"
cifar_dataset_path = "/Users/joaomacedo/LocalDev/datasets/cifar10"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# Hiperparâmetros
batch_size = 32 if device.type == 'cpu' else 64
num_epochs = 5
learning_rate = 0.001
image_size = 224  # Tamanho que a rede espera
num_workers = 0 if device.type == 'cpu' else 4

print(f"Batch size: {batch_size}")
print(f"Image size: {image_size}x{image_size}")
print(f"Epochs: {num_epochs}")
print(f"Learning rate: {learning_rate}")

# classes
classes_en = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
              'dog', 'frog', 'horse', 'ship', 'truck']
classes_br = ['aviao', 'carro', 'ave', 'gato', 'cervo', 
              'cachorro', 'sapo', 'cavalo', 'navio', 'caminhao']

In [ ]:
# define transformacoes dos dados
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# transformacoes para TREINO (com data augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),   # Resize para 224x224
    transforms.RandomHorizontalFlip(p=0.5),        # Flip horizontal aleatório
    transforms.ToTensor(),                         # Converte para tensor [0, 1]
    transforms.Normalize(mean=imagenet_mean,       # Normalização ImageNet
                        std=imagenet_std)
])

# transformacoes para VALIDAÇÃO/TESTE (sem augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),   # Apenas resize
    transforms.ToTensor(),                         # Converte para tensor [0, 1]
    transforms.Normalize(mean=imagenet_mean,       # Normalização ImageNet
                        std=imagenet_std)
])

print("Transformações de TREINO:")
print(f"  1. Resize: 32x32 -> {image_size}x{image_size}")
print(f"  2. RandomHorizontalFlip: 50% chance")
print(f"  3. ToTensor: [0, 255] -> [0, 1]")
print(f"  4. Normalize: ImageNet stats")

print("Transformações de VALIDAÇÃO/TESTE:")
print(f"  1. Resize: 32x32 -> {image_size}x{image_size}")
print(f"  2. ToTensor: [0, 255] -> [0, 1]")
print(f"  3. Normalize: ImageNet stats")


In [ ]:
# dataset de treino (com augmentation)
train_dataset_full = ImageFolder(
    root=f"{cifar_dataset_path}/train",
    transform=train_transforms
)

# dataset de teste (sem augmentation)
test_dataset = ImageFolder(
    root=f"{cifar_dataset_path}/test",
    transform=val_transforms
)

print(f"Dataset de treino carregado: {len(train_dataset_full)} imagens")
print(f"Dataset de teste carregado: {len(test_dataset)} imagens")
print(f"Classes detectadas: {train_dataset_full.classes}")


In [ ]:
# dividir particao de treino em treino e validacao

# calcular tamanhos
total_train = len(train_dataset_full)
train_size = int(0.9 * total_train)
val_size = total_train - train_size

# indices p dividir dataset
train_indices = list(range(train_size))
val_indices = list(range(train_size, total_train))

print(f"\nTotal de treino: {total_train}")
print(f"Treino (90%): {train_size}")
print(f"Validação (10%): {val_size}")

# dataset TREINO com augmentation
train_dataset = ImageFolder(
    root=f"{cifar_dataset_path}/train",
    transform=train_transforms
)

# dataset VALIDAÇÃO sem augmentation  
val_dataset_full = ImageFolder(
    root=f"{cifar_dataset_path}/train",
    transform=val_transforms
)

# restringe datasets aos indices
train_dataset = torch.utils.data.Subset(train_dataset, train_indices)
val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)



In [ ]:
# cria dataloaders

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == 'cuda')
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == 'cuda')
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == 'cuda')
)

print(f"DataLoaders criados:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

In [ ]:
# visualizar data augmentation

def denormalize(tensor, mean, std):
    """Desnormaliza tensor para visualização"""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean

# Pegar uma imagem e mostrar várias versões augmentadas
from PIL import Image
import os

# Pegar caminho de uma imagem
sample_class = train_dataset_full.classes[0]
sample_path = os.path.join(f"{cifar_dataset_path}/train", sample_class)

sample_image_path = os.path.join(sample_path, os.listdir(sample_path)[0])

# Aplicar transform múltiplas vezes
fig, axes = plt.subplots(2, 4, figsize=(6, 2.5))
fig.suptitle('Exemplos de Data Augmentation (Resize + Flip Horizontal)', 
             fontsize=14, fontweight='bold')

for idx in range(8):
    row = idx // 4
    col = idx % 4
    
    # Carregar e transformar
    img = Image.open(sample_image_path)
    img_tensor = train_transforms(img)
    
    # Desnormalizar para visualização
    img_denorm = denormalize(img_tensor, imagenet_mean, imagenet_std)
    img_denorm = torch.clamp(img_denorm, 0, 1)
    
    # Plotar
    axes[row, col].imshow(img_denorm.permute(1, 2, 0))
    axes[row, col].set_title(f"Amostra {idx+1}")
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("Cada vez que a imagem é carregada, o flip é aplicado aleatoriamente")

In [ ]:
# verificando shape apos transformacoes (antesde alimentar a rede)

# pegar um batch de treino
sample_batch_train = next(iter(train_loader))
images_train, labels_train = sample_batch_train

print(f"Batch de TREINO:")
print(f"  Images shape: {images_train.shape}")  # (batch, 3, 224, 224)
print(f"  Labels shape: {labels_train.shape}")  # (batch,)
print(f"  Images range: [{images_train.min():.3f}, {images_train.max():.3f}]")
print(f"  Images mean: {images_train.mean():.3f}")
print(f"  Images std: {images_train.std():.3f}")

# pegar um batch de validação
sample_batch_val = next(iter(val_loader))
images_val, labels_val = sample_batch_val

print(f"Batch de VALIDAÇÃO:")
print(f"  Images shape: {images_val.shape}")    # (batch, 3, 224, 224)
print(f"  Labels shape: {labels_val.shape}")    # (batch,)

print("Todas as imagens foram redimensionadas para 224x224")
print("Normalização ImageNet aplicada")


In [ ]:
# carregando modelo pre treinado

model = timm.create_model('mobilenetv3_small_100', pretrained=True, num_classes=10)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Modelo carregado: MobileNetV3-Small")
print(f"  Total de parâmetros: {total_params:,}")
print(f"  Parâmetros treináveis: {trainable_params:,}")
print(f"  Input esperado: (batch, 3, 224, 224)")
print(f"  Output: (batch, 10)")

# Testar forward pass
with torch.no_grad():
    test_output = model(images_train.to(device))
    print(f"Teste forward pass:")
    print(f"  Input: {images_train.shape}")
    print(f"  Output: {test_output.shape}")

In [ ]:
# config treinamento

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

print(f"Loss: CrossEntropyLoss")
print(f"Optimizer: AdamW (lr={learning_rate}, weight_decay=0.01)")
print(f"Scheduler: CosineAnnealingLR")


In [ ]:
# funcoes de treino e validacao
def train_epoch(model, dataloader, criterion, optimizer, device, epoch_num):
    """Treina o modelo por uma época"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f"Época {epoch_num} - Treino")
    
    for inputs, labels in pbar:
        # Inputs já vêm transformados: (batch, 3, 224, 224)
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        # Estatísticas
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Atualizar progress bar
        current_loss = running_loss / total
        current_acc = 100 * correct / total
        pbar.set_postfix({
            'loss': f'{current_loss:.4f}',
            'acc': f'{current_acc:.2f}%'
        })
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device, epoch_num, desc="Valid"):
    """Valida o modelo"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f"Época {epoch_num} - {desc}")
    
    with torch.no_grad():
        for inputs, labels in pbar:
            # Inputs já transformados
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            current_loss = running_loss / total
            current_acc = 100 * correct / total
            pbar.set_postfix({
                'loss': f'{current_loss:.4f}',
                'acc': f'{current_acc:.2f}%'
            })
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc



In [ ]:
# loop de treinamento

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_model_state = None

start_time = time.time()

for epoch in range(num_epochs):
    epoch_start = time.time()
    
    print(f"{'='*60}")
    print(f"Época {epoch+1}/{num_epochs}")
    print('='*60)
    
    # Treinar
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch_num=epoch+1
    )
    
    # Validar
    val_loss, val_acc = validate(
        model, val_loader, criterion, device, epoch_num=epoch+1, desc="Valid"
    )
    
    # Atualizar scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Salvar histórico
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Salvar melhor modelo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        print(f"Novo melhor modelo! Val Acc: {val_acc:.2f}%")
    
    # Tempo da época
    epoch_time = time.time() - epoch_start
    
    # Resumo
    print(f"Resumo Época {epoch+1}:")
    print(f"  Train -> Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  Val   -> Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
    print(f"  LR: {current_lr:.6f}")
    print(f"  Tempo: {epoch_time:.1f}s")
    
    # Early stopping
    if val_acc > 85:
        print(f"Acurácia alvo atingida! Parando treinamento.")
        break

total_time = time.time() - start_time

print(f"{'='*60}")
print("TREINAMENTO CONCLUÍDO!")
print(f"Tempo total: {total_time/60:.1f} minutos")
print(f"Tempo médio/época: {total_time/len(history['train_loss'])/60:.1f} minutos")
print(f"Melhor Val Acc: {best_val_acc:.2f}%")
print('='*60)